# AGFB Generator Visual Check

This notebook keeps the generator calls visible. The simplest pattern is `frame = generator(...)`, then `show_image(frame.I[0])`. Each generator returns `frame.I` with shape `(B, H, W)` and `frame.g` with shape `(B, 2, H, W)`.

In [ ]:
import math

import torch

from agfb_generators import (
    CompositeRect,
    composite,
    curved_arc,
    gaussian_blob,
    gaussian_ridge,
    hard_step,
    polynomial,
    sinusoid,
    smoothed_bar,
    smoothed_step,
)
from agfb_generators.notebook import set_color_scheme, setup_notebook, show_color_scheme, show_image

H, W = 160, 160
setup_notebook(height=H, width=W)

In [ ]:
COLOR_SCHEME = {
    "intensity": [(0.0, "#000000"), (0.55, "#73000A"), (1.0, "#FFFFFF")],
    "magnitude": [(0.0, "#000000"), (0.72, "#A49137"), (1.0, "#FFFFFF")],
    "signed": [(0.0, "#CC2E40"), (0.5, "#FFFFFF"), (1.0, "#466A9F")],
    "mask": [(0.0, "#000000"), (1.0, "#CED318")],
}

# Alternative grayscale scheme.
# COLOR_SCHEME = {
#     "intensity": [(0.0, "#000000"), (1.0, "#FFFFFF")],
#     "magnitude": [(0.0, "#000000"), (1.0, "#FFFFFF")],
#     "signed": [(0.0, "#000000"), (0.5, "#A2A2A2"), (1.0, "#FFFFFF")],
#     "mask": [(0.0, "#000000"), (1.0, "#FFFFFF")],
# }

# Alternative high-contrast warm/cool scheme.
# COLOR_SCHEME = {
#     "intensity": [(0.0, "#000000"), (0.5, "#73000A"), (1.0, "#FFF2E3")],
#     "magnitude": [(0.0, "#000000"), (0.65, "#65780B"), (1.0, "#FFFFFF")],
#     "signed": [(0.0, "#1F414D"), (0.5, "#FFFFFF"), (1.0, "#CC2E40")],
#     "mask": [(0.0, "#000000"), (1.0, "#CED318")],
# }

set_color_scheme(COLOR_SCHEME)
show_color_scheme(COLOR_SCHEME)

## Getting `g_x` And `g_y`

In [ ]:
frame = gaussian_blob(H, W, sigma=8, x0=-10, y0=8)

# These two lines are the direct gradient-channel access pattern.
gx = frame.g[0, 0]
gy = frame.g[0, 1]

# These aliases give the same tensors with a clearer name.
gx_alias = frame.gx[0]
gy_alias = frame.gy[0]

gmag = torch.sqrt(gx**2 + gy**2)
(
    frame.I.shape,
    frame.g.shape,
    gx.shape,
    gy.shape,
    torch.allclose(gx, gx_alias),
    torch.allclose(gy, gy_alias),
)

In [ ]:
show_image(frame.I[0], "gaussian_blob intensity")
show_image(gx, "gaussian_blob g_x", signed=True)
show_image(gy, "gaussian_blob g_y", signed=True)
show_image(gmag, "gaussian_blob |grad|", kind="magnitude")

## Current Generators

In [ ]:
frame = smoothed_step(H, W, theta_rad=math.radians(30), sigma_e=4)
show_image(frame.I[0], "smoothed_step")

In [ ]:
frame = hard_step(H, W, theta_rad=math.radians(15))
show_image(frame.I[0], "hard_step")

In [ ]:
frame = curved_arc(H, W, r0=42, sigma_e=4)
show_image(frame.I[0], "curved_arc")

In [ ]:
frame = sinusoid(H, W, freq=0.05, theta_rad=math.radians(30))
show_image(frame.I[0], "sinusoid")

In [ ]:
frame = gaussian_blob(H, W, sigma=8, x0=-10, y0=8)
show_image(frame.I[0], "gaussian_blob")

In [ ]:
frame = gaussian_ridge(H, W, sigma=4, theta_rad=math.radians(20))
show_image(frame.I[0], "gaussian_ridge")

In [ ]:
frame = smoothed_bar(H, W, width_px=32, theta_rad=math.radians(15), sigma_e=4)
show_image(frame.I[0], "smoothed_bar")

In [ ]:
coeffs = torch.zeros(1, 4, 4)
coeffs[0, 1, 0] = 0.3
coeffs[0, 0, 1] = -0.2
coeffs[0, 2, 1] = 0.05
coeffs[0, 1, 2] = -0.04

frame = polynomial(H, W, coeffs=coeffs, scale=64)
show_image(frame.I[0], "polynomial")

## A Composite Example

In [ ]:
a = smoothed_step(H, W, theta_rad=math.radians(30), sigma_e=4)
b = gaussian_blob(H, W, sigma=8, x0=-10, y0=8)
c = gaussian_ridge(H, W, sigma=4, theta_rad=math.radians(20))
d = sinusoid(H, W, freq=0.05, theta_rad=math.radians(30))

frame, mask = composite(
    H,
    W,
    [
        CompositeRect(0, H // 2, 0, W // 2, a),
        CompositeRect(0, H // 2, W // 2, W, b),
        CompositeRect(H // 2, H, 0, W // 2, c),
        CompositeRect(H // 2, H, W // 2, W, d),
    ],
)

show_image(frame.I[0], "composite")
show_image(mask.float(), "composite mask", kind="mask")

## A Batch Example

In [ ]:
frame = smoothed_step(
    H,
    W,
    theta_rad=torch.tensor([0.0, math.radians(30), math.radians(60)]),
    sigma_e=torch.tensor([2.0, 4.0, 6.0]),
)

show_image(frame.I[0], "batch item 0")
show_image(frame.I[1], "batch item 1")
show_image(frame.I[2], "batch item 2")